In [1]:
from google.colab import drive
drive.mount('/content/drive')

ZIP_PATH = "/content/drive/MyDrive/TFM/datasets/weapons_cs231n/raw_zip/weapons_yolov8_coco_added.zip"
WORK_DIR = "/content/weapons_det"
DATA_DIR = f"{WORK_DIR}/dataset_full"
PROJECT_DIR = "/content/drive/MyDrive/TFM/experiments/weapon_det"

import os, shutil, zipfile, glob

os.makedirs(WORK_DIR, exist_ok=True)
local_zip = f"{WORK_DIR}/dataset.zip"
shutil.copy(ZIP_PATH, local_zip)

if os.path.exists(DATA_DIR):
    shutil.rmtree(DATA_DIR)
os.makedirs(DATA_DIR, exist_ok=True)

with zipfile.ZipFile(local_zip, "r") as z:
    z.extractall(DATA_DIR)

# localizar data.yaml
candidates = glob.glob(f"{DATA_DIR}/**/data.yaml", recursive=True)
if not candidates:
    raise FileNotFoundError("No encuentro data.yaml en el zip descomprimido.")
yaml_path = candidates[0]
ROOT = os.path.dirname(yaml_path)

print("ROOT:", ROOT)
print("data.yaml:", yaml_path)

Mounted at /content/drive
ROOT: /content/weapons_det/dataset_full
data.yaml: /content/weapons_det/dataset_full/data.yaml


In [2]:
import os, glob

def get_val_split(root):
    if os.path.isdir(os.path.join(root, "valid")): return "valid"
    if os.path.isdir(os.path.join(root, "val")): return "val"
    raise FileNotFoundError("No existe valid/val")

VAL_SPLIT = get_val_split(ROOT)

def quick_label_stats(split, n_check=400):
    lbl_dir = os.path.join(ROOT, split, "labels")
    files = glob.glob(lbl_dir + "/*.txt")
    files = files[:min(n_check, len(files))]
    empty = 0
    nonempty = 0
    for f in files:
        with open(f, "r") as fh:
            lines = [ln.strip() for ln in fh.readlines() if ln.strip()]
        if lines: nonempty += 1
        else: empty += 1
    return len(files), empty, nonempty

for sp in ["train", VAL_SPLIT]:
    total, empty, nonempty = quick_label_stats(sp)
    print(f"{sp}: checked={total} empty={empty} nonempty={nonempty}")

train: checked=400 empty=74 nonempty=326
valid: checked=400 empty=0 nonempty=400


In [3]:
!pip -q install ultralytics

from ultralytics import YOLO

model = YOLO("yolov8m.pt")

model.train(
    data=yaml_path,
    epochs=50,
    imgsz=640,
    batch=-1,         # auto-batch: muy útil en Pro+ para exprimir la GPU sin OOM
    workers=4,
    device=0,
    project=PROJECT_DIR,
    name="yolov8m_weapons_B_e50_640",   # B = dataset con negativos COCO
    exist_ok=True
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 66.6 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/weapons_det/dataset_full/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, 

ultralytics.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0])
box: ultralytics.utils.metrics.Metric object
confusion_matrix: <ultralytics.utils.metrics.ConfusionMatrix object at 0x7a25191cbad0>
curves: ['Precision-Recall(B)', 'F1-Confidence(B)', 'Precision-Confidence(B)', 'Recall-Confidence(B)']
curves_results: [[array([          0,    0.001001,    0.002002,    0.003003,    0.004004,    0.005005,    0.006006,    0.007007,    0.008008,    0.009009,     0.01001,    0.011011,    0.012012,    0.013013,    0.014014,    0.015015,    0.016016,    0.017017,    0.018018,    0.019019,     0.02002,    0.021021,    0.022022,    0.023023,
          0.024024,    0.025025,    0.026026,    0.027027,    0.028028,    0.029029,     0.03003,    0.031031,    0.032032,    0.033033,    0.034034,    0.035035,    0.036036,    0.037037,    0.038038,    0.039039,     0.04004,    0.041041,    0.042042,    0.043043,    0.044044,    0.045045,    0.046046,    0.047047,
          0.048048, 